Project Completed by **Ahmed Salama**
* [GitHub](https://github.com/ahmedsalama00)
* [Linked In](https://www.linkedin.com/in/ahmedsalamaa00/)
* [My Portfolio ](https://ahmedsalama00.github.io/Ahmed)
* [Movie Recommender System Repo](https://github.com/ahmedsalama00/Movie-Recommender-System)

### Phase 1: Environment Setup and Data Loading

In [1]:
import os
import pandas as pd
import numpy as np
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.decomposition import TruncatedSVD
from sklearn.metrics import precision_score, recall_score

df_movies = pd.read_csv('movie.csv')
df_movies['genres'] = df_movies['genres'].fillna('')

rating_path = 'rating.csv'
if not os.path.exists(rating_path):
    rating_path = 'D:/Machine Learning Uneeq Interns/Recommendation System/rating.csv'

df_ratings = pd.read_csv(rating_path, nrows=100000)

### Phase 2: Feature Engineering

In [2]:
tfidf = TfidfVectorizer(token_pattern=r'[^|]+')
tfidf_matrix = tfidf.fit_transform(df_movies['genres'])
user_movie_matrix = df_ratings.pivot(index='userId', columns='movieId', values='rating').fillna(0)
svd = TruncatedSVD(n_components=50, random_state=42)
svd_matrix = svd.fit_transform(user_movie_matrix.T)

### Phase 3: Building the Recommendation Engine

In [4]:
def get_recommendations(df_movies, tfidf_matrix, svd_matrix, user_movie_matrix, title, top_n=5):
    if title not in df_movies['title'].values:
        return []
    
    idx = df_movies[df_movies['title'] == title].index[0]
    movie_id = df_movies.iloc[idx]['movieId']
    
    sim_scores_content = cosine_similarity(tfidf_matrix[idx], tfidf_matrix).flatten()
    sim_scores_collab = np.zeros(len(df_movies))
    
    if movie_id in user_movie_matrix.columns:
        collab_idx = user_movie_matrix.columns.get_loc(movie_id)
        collab_sim = cosine_similarity([svd_matrix[collab_idx]], svd_matrix).flatten()
        collab_series = pd.Series(collab_sim, index=user_movie_matrix.columns)
        sim_scores_collab = df_movies['movieId'].map(collab_series).fillna(0).values
        
    hybrid_scores = 0.5 * sim_scores_content + 0.5 * sim_scores_collab
    
    sim_scores = list(enumerate(hybrid_scores))
    sim_scores = sorted(sim_scores, key=lambda x: x[1], reverse=True)
    sim_scores = sim_scores[1:top_n+1]
    
    return [df_movies.iloc[i[0]]['title'] for i in sim_scores]

### Phase 4: Evaluation

In [6]:
def build_ground_truth(df_ratings, df_movies, threshold=4.0):
    user_likes = {}
    subset_users = df_ratings['userId'].unique()[:50]
    for user in subset_users:
        liked_movie_ids = df_ratings[(df_ratings['userId'] == user) & (df_ratings['rating'] >= threshold)]['movieId'].tolist()
        liked_titles = df_movies[df_movies['movieId'].isin(liked_movie_ids)]['title'].tolist()
        if liked_titles:
            user_likes[user] = liked_titles
    return user_likes

def evaluate(df_movies, tfidf_matrix, svd_matrix, user_movie_matrix, ground_truth, top_n=5):
    y_true = []
    y_pred = []
    
    for user, liked_titles in list(ground_truth.items()):
        for title in liked_titles[:3]:
            if title not in df_movies['title'].values:
                continue
            preds = get_recommendations(df_movies, tfidf_matrix, svd_matrix, user_movie_matrix, title, top_n=top_n)
            trues = set(liked_titles) - {title}
            
            y_true.extend([1 if item in trues else 0 for item in preds])
            y_pred.extend([1] * len(preds))
            
    if len(y_true) == 0:
        return 0.0, 0.0
    precision = precision_score(y_true, y_pred, zero_division=0)
    recall = recall_score(y_true, y_pred, zero_division=0)
    return precision, recall

ground_truth = build_ground_truth(df_ratings, df_movies)
precision, recall = evaluate(df_movies, tfidf_matrix, svd_matrix, user_movie_matrix, ground_truth, top_n=5)
print(f"Precision: {precision:.2f}")
print(f"Recall: {recall:.2f}")

Precision: 0.14
Recall: 1.00


### Phase 5: Inference

In [7]:
movie_name = "Toy Story "
recs = get_recommendations(df_movies, tfidf_matrix, svd_matrix, user_movie_matrix, movie_name, top_n=5)
print(f"Recommendation for film {movie_name}:")
for i, m in enumerate(recs, 1):
    print(f"{i}. {m}")

Recommendation for film Toy Story :
1. Toy Story 2 
2. Monsters, Inc. 
3. Shrek 
4. Bug's Life, A 
5. Antz 
